In [4]:
from mdcrow import MDCrow
from langchain.callbacks import get_openai_callback

In [5]:
def calculate_llm_cost(input_tokens, output_tokens, model):
    pricing_2024 = {
        "gpt-4-1106-preview": {"input": 10/1e6, "output": 30/1e6},
        "gpt-3.5-turbo-0125": {"input": 0.5/1e6, "output": 1.5/1e6},
        "gpt-4-turbo-2024-04-09": {"input": 10/1e6, "output": 30/1e6},
        "gpt-4o-2024-08-06": {"input": 5/1e6, "output": 15/1e6},
        "llama-v3p1-70b-instruct": {"input": 0.9/1e6, "output": 0.9/1e6},
        "llama-v3p1-405b-instruct": {"input": 3/1e6, "output": 3/1e6},
        "claude-3-opus": {"input": 15/1e6, "output": 75/1e6},
        "claude-3.5-sonnet": {"input": 3/1e6, "output": 15/1e6},
    }
    pricing_2025 = {
        "gpt-4-1106-preview": {"input": 10/1e6, "output": 30/1e6},
        "gpt-3.5-turbo-0125": {"input": 0.5/1e6, "output": 1.5/1e6},
        "gpt-4-turbo-2024-04-09": {"input": 10/1e6, "output": 30/1e6},
        "gpt-4o-2024-08-06": {"input": 2.5/1e6, "output": 10/1e6},
        "llama-v3p1-70b-instruct": {"input": 0.9/1e6, "output": 0.9/1e6},
        "llama-v3p1-405b-instruct": {"input": 3/1e6, "output": 3/1e6},
    }
    
    prices = pricing_2025[model]
    cost = (input_tokens * prices["input"]) + (output_tokens * prices["output"])
    return round(cost, 6)

In [6]:
llm_model = "gpt-4o-2024-08-06"
tools = "all"

agent = MDCrow(
    agent_type="Structured", 
    model=llm_model, 
    top_k_tools=tools,
    streaming=False,
    verbose=True
)

In [ ]:
prompt = """Download PDB 1ZNI. Check its dimensions and recommend a cubic box 
with 1.0 nm padding on all sides. Verify the box is sufficient. 
Once confirmed, simulate at 300 K for 1 ps and calculate RMSD."""

with get_openai_callback() as cb:
    print(agent.run(prompt, callbacks=[cb]))
    print(cb)

In [7]:
# Download, then ask about dimensions (explicit instruction)
level_1_prompt = """Download PDB 1L2Y. 
What are the approximate dimensions of the protein (width in X, Y, Z)?
Explain how you determined this."""

# ask about padding (introduce concept)
level_2_prompt = """Download PDB 1L2Y.
What would be an appropriate box size for this protein?
Consider that proteins typically need 1.0 nm padding on all sides.
Explain your reasoning."""

# request validation reasoning (agent must evaluate)
level_3_prompt = """Download PDB 1L2Y.
Recommend a cubic box size with 1.0 nm padding.
Is this box size appropriate? Explain what would make it too small or too large."""

# Full solvent estimation (requires calculation chain)
level_4_prompt = """Download PDB 1L2Y.
Recommend a cubic box size with 1.0 nm padding.
Can we estimate how many water molecules would be needed to solvate this system? 
Explain your approach."""

# Compare geometries (decision-making)
level_5_prompt = """Download PDB 1L2Y.
Compare cubic vs rectangular box geometries for this protein.
Which would be more efficient? Why?
Recommend one and explain."""

# Edge case reasoning (elongated protein consideration)
level_6_prompt = """Download PDB 1L2Y.
Calculate its aspect ratio (longest dimension / shortest dimension).
If this protein were more elongated (e.g., aspect ratio > 3), 
how would that change your box size recommendation?
Explain."""

In [8]:
agent = MDCrow(
    agent_type="Structured", 
    model=llm_model, 
    top_k_tools=tools,
    streaming=False,
    verbose=True
)
print(agent.llm.model_name)

with get_openai_callback() as cb:
    print(agent.run(level_1_prompt, callbacks=[cb]))
    print(cb)
print('LLM cost:',calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name))

gpt-4o-2024-08-06


> Entering new AgentExecutor chain...
Thought: To determine the approximate dimensions of the protein in the PDB file 1L2Y, I first need to download the PDB file. Once downloaded, I can analyze the file to find the dimensions in the X, Y, and Z directions. 

Action:
```
{
  "action": "PDBFileDownloader",
  "action_input": "1L2Y"
}
```PDB file found with this ID: 1L2Y

Observation: Succeeded. Downloaded the PDB file:1L2Y_230458
Thought:To determine the approximate dimensions of the protein in the PDB file 1L2Y, I need to analyze the downloaded PDB file to find the dimensions in the X, Y, and Z directions. I will use the appropriate tool to analyze the structure and extract the dimensions.

Action:
```
{
  "action": "SummarizeProteinStructure",
  "action_input": {
    "traj_file": "1L2Y_230458"
  }
}
```

/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(



Observation: {'n_atoms': 304, 'n_residues': 20, 'n_chains': 1, 'n_frames': 38, 'n_bonds': 310}
Thought:To determine the approximate dimensions of the protein in the PDB file 1L2Y, I need to calculate the bounding box dimensions in the X, Y, and Z directions. I will use the appropriate tool to analyze the structure and extract these dimensions.

Action:
```
{
  "action": "MomentOfInertia",
  "action_input": {
    "top_fileid": "1L2Y_230458"
  }
}
```Calculating moments of inertia...
Data saved to /Users/quinny/repos/MDCrow/ckpt/ckpt_48/records/MOI_1L2Y_230458.csv


/Users/quinny/repos/MDCrow/mdcrow/utils/data_handling.py:49: UserWarning: Trajectory File ID is not provided but is not required; loading MDTrajectory from topology 1L2Y_230458 only.
  warnings.warn(
/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


Plot of moments of inertia saved to FIG_MOI_1L2Y_230458_20251222_230502.png

Observation: Succeeded. Data saved with file ID MOI_1L2Y_230458. 
Average Moment of Inertia of all frames: 762.89. 
Plot of moments of inertia over time saved with plot ID fig0_230502. 

Thought:To determine the approximate dimensions of the protein in the PDB file 1L2Y, I need to calculate the bounding box dimensions in the X, Y, and Z directions. I will use the appropriate tool to analyze the structure and extract these dimensions.

Action:
```
{
  "action": "RadiusofGyrationTool",
  "action_input": {
    "traj_file": "1L2Y_230458",
    "top_file": "1L2Y_230458"
  }
}
```plot_name:  FIG_rgy_1L2Y_230458_20251222_230504.png
plot_path /Users/quinny/repos/MDCrow/ckpt/ckpt_48/figures/FIG_rgy_1L2Y_230458_20251222_230504.png


...

/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


Could not parse LLM output: To determine the approximate dimensions of the protein in the PDB file 1L2Y, I have successfully computed the gyration tensor for the first frame. The gyration tensor provides information about the distribution of mass around the center of mass of the protein, which can be used to estimate the dimensions in the X, Y, and Z directions.

The gyration tensor for the first frame is:
```
[[ 0.2651748  -0.04295195  0.0305956 ]
 [-0.04295195  0.20008835  0.01212866]
 [ 0.0305956   0.01212866  0.09324988]]
```

The diagonal elements of the gyration tensor represent the distribution of mass along the principal axes, which can be interpreted as the approximate dimensions of the protein in the X, Y, and Z directions. Therefore, the approximate dimensions of the protein are:
- X: 0.265 nm
- Y: 0.200 nm
- Z: 0.093 nm

These values provide an estimate of the protein's size in each direction based on the distribution of its atoms.

Final Answer: The approximate dimensions 

/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(



Observation: Gyration tensor computed successfully, saved to rec0_230535.Multiple frames found. Here's result for 1st frame: [[ 0.2651748  -0.04295195  0.0305956 ]
 [-0.04295195  0.20008835  0.01212866]
 [ 0.0305956   0.01212866  0.09324988]]
Thought:Question: What are the approximate dimensions of the protein in the PDB file 1L2Y based on the gyration tensor?
Thought: I need to provide the final answer in the specified JSON format.
Action:
```
{
  "action": "Final Answer",
  "action_input": "The approximate dimensions of the protein in PDB file 1L2Y are X: 0.265 nm, Y: 0.200 nm, and Z: 0.093 nm, determined from the gyration tensor of the first frame."
}
```

> Finished chain.
({'input': '\n    You are an expert molecular dynamics scientist, and\n    your task is to respond to the question or\n    solve the problem to the best of your ability using\n    the provided tools.\n\n    You can only respond with a single complete\n    \'Thought, Action, Action Input\' format\n    OR a single

<Figure size 640x480 with 0 Axes>

In [9]:
with get_openai_callback() as cb:
    print(agent.run(level_2_prompt, callbacks=[cb]))
    print(cb)
print('LLM cost:',calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name))



> Entering new AgentExecutor chain...
Thought: To determine an appropriate box size for the protein with PDB ID 1L2Y, I first need to download the PDB file to analyze its dimensions. Once I have the dimensions, I can add a 1.0 nm padding on all sides to determine the box size. 

Action:
```
{
  "action": "PDBFileDownloader",
  "action_input": "1L2Y"
}
```PDB file found with this ID: 1L2Y

Observation: Succeeded. Downloaded the PDB file:1L2Y_230540
Thought:To determine the appropriate box size for the protein with PDB ID 1L2Y, I need to analyze the dimensions of the protein structure. Once I have the dimensions, I can add a 1.0 nm padding on all sides to determine the box size.

Action:
```
{
  "action": "MomentOfInertia",
  "action_input": {
    "top_fileid": "1L2Y_230540",
    "traj_fileid": "",
    "molecule_name": "1L2Y"
  }
}
```
Observation: Failed. OSError: Sorry, no loader for filename=Name not found in path registry. (extension=.) was found. I can only load files with extensi

/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


To determine the appropriate box size for the protein with PDB ID 1L2Y, I need to analyze the dimensions of the protein structure. Since the previous attempt to use the "MomentOfInertia" tool failed due to a file extension issue, I will try a different approach to obtain the dimensions of the protein. Once I have the dimensions, I can add a 1.0 nm padding on all sides to determine the box size.

Action:
```
{
  "action": "SummarizeProteinStructure",
  "action_input": {
    "traj_file": "1L2Y_230540"
  }
}
```
Observation: {'n_atoms': 304, 'n_residues': 20, 'n_chains': 1, 'n_frames': 38, 'n_bonds': 310}
Thought:

/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


To determine the appropriate box size for the protein with PDB ID 1L2Y, I need to calculate the dimensions of the protein structure. The "SummarizeProteinStructure" tool provided information about the number of atoms, residues, chains, frames, and bonds, but not the spatial dimensions. I need to use a different approach to obtain the spatial dimensions of the protein.

Action:
```
{
  "action": "RadiusofGyrationTool",
  "action_input": {
    "traj_file": "1L2Y_230540",
    "top_file": "1L2Y_230540"
  }
}
```
plot_name:  FIG_rgy_1L2Y_230540_20251222_230547.png
plot_path /Users/quinny/repos/MDCrow/ckpt/ckpt_48/figures/FIG_rgy_1L2Y_230540_20251222_230547.png


/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(



Observation: Succeeded. Radii of gyration saved to /Users/quinny/repos/MDCrow/ckpt/ckpt_48/figures/radii_of_gyration_1L2Y_230540.csv with id rgy_1L2Y_230540.Plot saved as: FIG_rgy_1L2Y_230540_20251222_230547.png with plot ID fig0_230547Average radius of gyration: 0.74 nm
Thought:To determine the appropriate box size for the protein with PDB ID 1L2Y, I have successfully obtained the average radius of gyration, which is 0.74 nm. The radius of gyration gives an estimate of the protein's size, but to determine the box size, we need to consider the maximum extent of the protein.

Typically, the box size should be large enough to accommodate the protein with additional padding to prevent interactions with its periodic images. A common practice is to add a padding of 1.0 nm on all sides of the protein.

Given the average radius of gyration is 0.74 nm, the diameter (or maximum extent) of the protein can be roughly estimated as twice the radius of gyration, which is approximately 1.48 nm. Addi

<Figure size 640x480 with 0 Axes>

In [10]:
with get_openai_callback() as cb:
    print(agent.run(level_3_prompt, callbacks=[cb]))
    print(cb)
print('LLM cost:',calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name))



> Entering new AgentExecutor chain...
Thought: First, I need to download the PDB file for 1L2Y. Then, I will calculate the dimensions of the protein to recommend a cubic box size with 1.0 nm padding. Finally, I will evaluate if the box size is appropriate and explain the factors that could make it too small or too large.

Action:
```json
{
  "action": "PDBFileDownloader",
  "action_input": "1L2Y"
}
```PDB file found with this ID: 1L2Y

Observation: Succeeded. Downloaded the PDB file:1L2Y_230610
Thought:To recommend a cubic box size with 1.0 nm padding, I need to calculate the dimensions of the protein in the PDB file 1L2Y. This will allow me to determine the appropriate box size. After calculating the dimensions, I will add 1.0 nm padding to each side to ensure the protein fits comfortably within the box. Then, I will evaluate if the box size is appropriate and explain the factors that could make it too small or too large.

Action:
```json
{
  "action": "SummarizeProteinStructure",
 

/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(



Observation: {'n_atoms': 304, 'n_residues': 20, 'n_chains': 1, 'n_frames': 38, 'n_bonds': 310}
Thought:To recommend a cubic box size with 1.0 nm padding, I need to calculate the dimensions of the protein in the PDB file 1L2Y. This will allow me to determine the appropriate box size. After calculating the dimensions, I will add 1.0 nm padding to each side to ensure the protein fits comfortably within the box. Then, I will evaluate if the box size is appropriate and explain the factors that could make it too small or too large.

Action:
```json
{
  "action": "RadiusofGyrationTool",
  "action_input": {
    "traj_file": "1L2Y_230610",
    "top_file": "1L2Y_230610"
  }
}
```plot_name:  FIG_rgy_1L2Y_230610_20251222_230613.png
plot_path /Users/quinny/repos/MDCrow/ckpt/ckpt_48/figures/FIG_rgy_1L2Y_230610_20251222_230613.png


/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(



Observation: Succeeded. Radii of gyration saved to /Users/quinny/repos/MDCrow/ckpt/ckpt_48/figures/radii_of_gyration_1L2Y_230610.csv with id rgy_1L2Y_230610.Plot saved as: FIG_rgy_1L2Y_230610_20251222_230613.png with plot ID fig0_230613Average radius of gyration: 0.74 nm
Thought:To determine the appropriate cubic box size with 1.0 nm padding, I need to calculate the dimensions of the protein using the average radius of gyration. The average radius of gyration for the protein is 0.74 nm. To ensure the protein fits comfortably within the box, I will add 1.0 nm padding to each side.

The formula for the box size is:
\[ \text{Box Size} = 2 \times (\text{Radius of Gyration} + \text{Padding}) \]

Let's calculate the box size:

\[ \text{Box Size} = 2 \times (0.74 \, \text{nm} + 1.0 \, \text{nm}) = 3.48 \, \text{nm} \]

Now, let's evaluate if this box size is appropriate:

1. **Too Small**: If the box size is too small, the protein may interact with its periodic images, leading to artifacts i

<Figure size 640x480 with 0 Axes>

In [11]:
with get_openai_callback() as cb:
    print(agent.run(level_5_prompt, callbacks=[cb]))
    print(cb)
print('LLM cost:',calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name))



> Entering new AgentExecutor chain...
Thought: To address the question, I need to first download the PDB file for the protein 1L2Y. Then, I will compare cubic and rectangular box geometries for this protein in terms of efficiency. Efficiency in molecular dynamics simulations often refers to computational efficiency and the ability to accurately simulate the system with minimal artifacts. I will start by downloading the PDB file.

Action:
```json
{
  "action": "PDBFileDownloader",
  "action_input": "1L2Y"
}
```PDB file found with this ID: 1L2Y

Observation: Succeeded. Downloaded the PDB file:1L2Y_230632
Thought:To compare cubic and rectangular box geometries for the protein 1L2Y, I need to consider the computational efficiency and accuracy of simulations using these geometries. The efficiency can be influenced by factors such as the number of water molecules required to solvate the protein and the periodic boundary conditions applied.

1. **Cubic Box**: 
   - A cubic box is symmetric 

In [12]:
with get_openai_callback() as cb:
    print(agent.run(level_6_prompt, callbacks=[cb]))
    print(cb)
print('LLM cost:',calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name))



> Entering new AgentExecutor chain...
Thought: To address the question, I need to first download the PDB file for 1L2Y. Then, I will calculate its aspect ratio by determining the longest and shortest dimensions. Finally, I will provide an explanation on how an elongated aspect ratio would affect the box size recommendation.

Action:
```
{
  "action": "PDBFileDownloader",
  "action_input": "1L2Y"
}
```PDB file found with this ID: 1L2Y

Observation: Succeeded. Downloaded the PDB file:1L2Y_230653
Thought:To calculate the aspect ratio, I need to determine the longest and shortest dimensions of the protein structure from the downloaded PDB file. I will use the MomentOfInertia tool to compute the moment of inertia tensors, which will help in determining these dimensions.

Action:
```json
{
  "action": "MomentOfInertia",
  "action_input": {
    "top_fileid": "1L2Y_230653",
    "traj_fileid": "",
    "molecule_name": "1L2Y"
  }
}
```
Observation: Failed. OSError: Sorry, no loader for filenam

/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


To proceed, I need to ensure that the file extension is correct for the tools to process it. The downloaded file should have a valid extension like `.pdb`. I will list the registry paths to confirm the file's presence and its correct extension.

Action:
```json
{
  "action": "ListRegistryPaths",
  "action_input": "paths"
}
```
Observation: Succeeded. Files found in registry: 1L2Y_230458: PDB file downloaded from RSCB, PDBFile ID: 1L2Y_230458, MOI_1L2Y_230458: Moments of inertia for 1L2Y_230458, fig0_230502: Plot of moments of inertia over time for 1L2Y_230458, rgy_1L2Y_230458: Radii of gyration per frame for 1L2Y_230458, fig0_230504: Plot of radii of gyration over time for 1L2Y_230458, rec0_230506: gyration_tensor values for trajectory with id: 1L2Y_230458, rec0_230523: gyration_tensor values for trajectory with id: 1L2Y_230458, rec0_230528: gyration_tensor values for trajectory with id: 1L2Y_230458, rec0_230535: gyration_tensor values for trajectory with id: 1L2Y_230458, 1L2Y_230540: 

/Users/quinny/repos/MDCrow/mdcrow/utils/data_handling.py:49: UserWarning: Trajectory File ID is not provided but is not required; loading MDTrajectory from topology 1L2Y_230653 only.
  warnings.warn(
/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


Calculating moments of inertia...
Data saved to /Users/quinny/repos/MDCrow/ckpt/ckpt_48/records/MOI_1L2Y.csv
Plot of moments of inertia saved to FIG_MOI_1L2Y_20251222_230701.png

Observation: Succeeded. Data saved with file ID MOI_1L2Y. 
Average Moment of Inertia of all frames: 762.89. 
Plot of moments of inertia over time saved with plot ID fig0_230701. 

Thought:To calculate the aspect ratio, I need to determine the longest and shortest dimensions of the protein structure using the moment of inertia data. The moment of inertia provides information about the distribution of mass in the protein, which can be used to infer these dimensions. Once I have these dimensions, I can calculate the aspect ratio and provide an explanation on how an elongated aspect ratio would affect the box size recommendation.

Action:
```json
{
  "action": "ComputeGyrationTensor",
  "action_input": {
    "traj_file": "MOI_1L2Y",
    "top_file": "1L2Y_230653"
  }
}
```

Observation: Sorry, no loader for filenam

/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


To proceed with calculating the aspect ratio, I need to determine the longest and shortest dimensions of the protein structure. Since the `MomentOfInertia` tool provided the average moment of inertia, I should use the `ComputeGyrationTensor` tool to obtain the gyration tensor, which will help in determining these dimensions. However, it seems there was an issue with the file format. I will attempt to use the `ComputeGyrationTensor` tool again with the correct file ID.

Action:
```json
{
  "action": "ComputeGyrationTensor",
  "action_input": {
    "traj_file": "1L2Y_230653",
    "top_file": "1L2Y_230653"
  }
}
```
Observation: Gyration tensor computed successfully, saved to rec0_230707.Multiple frames found. Here's result for 1st frame: [[ 0.2651748  -0.04295195  0.0305956 ]
 [-0.04295195  0.20008835  0.01212866]
 [ 0.0305956   0.01212866  0.09324988]]
Thought:

/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


Could not parse LLM output: To calculate the aspect ratio, I need to determine the longest and shortest dimensions of the protein structure using the gyration tensor. The gyration tensor provides information about the distribution of mass in the protein, which can be used to infer these dimensions. Once I have these dimensions, I can calculate the aspect ratio and provide an explanation on how an elongated aspect ratio would affect the box size recommendation.

The gyration tensor for the first frame is:
```
[[ 0.2651748  -0.04295195  0.0305956 ]
 [-0.04295195  0.20008835  0.01212866]
 [ 0.0305956   0.01212866  0.09324988]]
```

The eigenvalues of this tensor represent the principal moments of inertia, which correspond to the squared radii of gyration along the principal axes. The square root of these eigenvalues gives the dimensions of the protein along these axes.

Let's calculate the eigenvalues to determine the longest and shortest dimensions.
Observation: Invalid or incomplete res

ValueError: unknown format from LLM: To solve this problem, we need to calculate the square roots of the eigenvalues of a given tensor (matrix). We will use Python's `numpy` library to compute the eigenvalues and then take their square roots. Here's how you can translate this into a Python expression using `numexpr`:

First, let's break down the problem:

1. We have a 3x3 matrix (tensor).
2. We need to compute the eigenvalues of this matrix.
3. We then take the square root of each eigenvalue.

Here's the step-by-step solution:

```python
import numpy as np
import numexpr as ne

# Define the tensor (matrix)
tensor = np.array([
    [0.2651748, -0.04295195, 0.0305956],
    [-0.04295195, 0.20008835, 0.01212866],
    [0.0305956, 0.01212866, 0.09324988]
])

# Compute the eigenvalues
eigenvalues = np.linalg.eigvalsh(tensor)

# Compute the square roots of the eigenvalues
sqrt_eigenvalues = np.sqrt(eigenvalues)

# Evaluate using numexpr
result = ne.evaluate("sqrt_eigenvalues")

print(result)
``

In [13]:
prompt = """Download PDB 1L2Y. 
Use ComputeGyrationTensor tool on it. 
Did you find any helpful information on estimating box size? 
No need to run any other tools. """
with get_openai_callback() as cb:
    print(agent.run(prompt, callbacks=[cb]))
    print(cb)
print('LLM cost:',calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name))



> Entering new AgentExecutor chain...
Thought: To address the question, I need to first download the PDB file for 1L2Y. Then, I will use the ComputeGyrationTensor tool on the downloaded file. This will help in estimating the box size. Let's start by downloading the PDB file.

Action:
```json
{
  "action": "PDBFileDownloader",
  "action_input": "1L2Y"
}
```PDB file found with this ID: 1L2Y

Observation: Succeeded. Downloaded the PDB file:1L2Y_230908
Thought:To proceed, I will use the ComputeGyrationTensor tool on the downloaded PDB file (1L2Y_230908) to analyze the gyration tensor, which can provide insights into estimating the box size.

Action:
```json
{
  "action": "ComputeGyrationTensor",
  "action_input": {
    "traj_file": "1L2Y_230908",
    "top_file": "1L2Y_230908"
  }
}
```

/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(



Observation: Gyration tensor computed successfully, saved to rec0_230910.Multiple frames found. Here's result for 1st frame: [[ 0.2651748  -0.04295195  0.0305956 ]
 [-0.04295195  0.20008835  0.01212866]
 [ 0.0305956   0.01212866  0.09324988]]
Thought:The gyration tensor provides information about the distribution of mass around the center of mass of the molecule. It can be used to estimate the size and shape of the molecule, which is helpful in determining the appropriate box size for simulations. The diagonal elements of the tensor represent the spread of the molecule along the principal axes, while the off-diagonal elements indicate the correlation between these axes.

For estimating the box size, you can consider the largest diagonal element of the gyration tensor as an indicator of the molecule's maximum extent in any direction. This value can be used to set the minimum box dimension, ensuring that the molecule has enough space to move without interacting with its periodic images.

In [15]:
prompt = """Download PDB 1L2Y. Recommend box size with 1.0 nm padding, 
and justify why this is sufficient. Estimate the solvent requirement. 
Then simulate at 300 K for 100 steps and calculate RMSD.""" # can go up to 10 ps

with get_openai_callback() as cb:
    print(agent.run(prompt, callbacks=[cb]))
    print(cb)
print('LLM cost:',calculate_llm_cost(cb.prompt_tokens, cb.completion_tokens, agent.llm.model_name))



> Entering new AgentExecutor chain...
Thought: To address the question, I need to perform several tasks: download the PDB file for 1L2Y, determine an appropriate box size with a 1.0 nm padding, estimate the solvent requirement, set up and run a short simulation at 300 K for 100 steps, and finally calculate the RMSD. I'll start by downloading the PDB file for 1L2Y.

Action:
```
{
  "action": "PDBFileDownloader",
  "action_input": "1L2Y"
}
```PDB file found with this ID: 1L2Y

Observation: Succeeded. Downloaded the PDB file:1L2Y_231007
Thought:To proceed, I need to determine the appropriate box size with a 1.0 nm padding around the protein structure in the downloaded PDB file. After that, I will estimate the solvent requirement. 

Action:
```
{
  "action": "SummarizeProteinStructure",
  "action_input": {
    "traj_file": "1L2Y_231007"
  }
}
```
Observation: {'n_atoms': 304, 'n_residues': 20, 'n_chains': 1, 'n_frames': 38, 'n_bonds': 310}
Thought:

/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


To determine the appropriate box size with a 1.0 nm padding, I need to calculate the dimensions of the protein and then add the padding. This will help in estimating the solvent requirement as well. Let's proceed with calculating the dimensions of the protein.

Action:
```
{
  "action": "ComputeGyrationTensor",
  "action_input": {
    "traj_file": "1L2Y_231007"
  }
}
```
Observation: Gyration tensor computed successfully, saved to rec0_231010.Multiple frames found. Here's result for 1st frame: [[ 0.2651748  -0.04295195  0.0305956 ]
 [-0.04295195  0.20008835  0.01212866]
 [ 0.0305956   0.01212866  0.09324988]]
Thought:

/opt/anaconda3/envs/mdcrow/lib/python3.12/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


Question: How do I determine the appropriate box size with a 1.0 nm padding using the gyration tensor?

Thought: I need to calculate the root mean square (RMS) dimensions along each principal axis from the diagonal elements of the gyration tensor and then add the 1.0 nm padding to determine the box size.

Action:
```
{
  "action": "Calculator",
  "action_input": "sqrt(0.2651748) + 1.0, sqrt(0.20008835) + 1.0, sqrt(0.09324988) + 1.0"
}
```

ValueError: LLMMathChain._evaluate("
sqrt(0.2651748) + 1.0, sqrt(0.20008835) + 1.0, sqrt(0.09324988) + 1.0
") raised error: unsupported expression type: <class 'tuple'>. Please try again with a valid numerical expression